# Fig. 2 / Fig. 8 (App. B.4) heatmaps — reproduction

Generates the two headline heatmaps: Fig. 2 (six fully-covered prompting
strategies) and the Fig. 8 appendix heatmap (App. B.4, all eleven). Both are built on
`binding_gate.py`'s fact gate (the same one every other script in this folder
uses), so the population and every N here matches Tables 7–12 by construction.

This notebook **supersedes `evaluation_notebook_complete.ipynb`** for
reproducing Fig. 2 / App. B.4: that notebook predates the fact-gate module and
re-derives the known population its own way. Keep it if you want the record
of what produced earlier figures, but use this one going forward.

**Run this from the folder that contains** `binding_failures_athletes_rebuttal_temp0.3/`
and `binding_failures_custom_rebuttal_temp0.3/`. Cell 2 below points Python at
this file's own directory so `import binding_gate` works regardless of where
Jupyter's own working directory happens to be.


In [ ]:
%matplotlib inline
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import Normalize

# Point Python at this notebook's own directory (where binding_gate.py lives),
# regardless of what directory Jupyter itself was launched from.
RESULTS_DIR = Path.cwd()  # edit if this notebook is not opened from its own folder
if str(RESULTS_DIR) not in sys.path:
    sys.path.insert(0, str(RESULTS_DIR))

from binding_gate import (
    ALL_BINDING_TYPES, DATASETS, MODEL_LABELS, MODEL_DIRS, ROWS, STAR_MODELS,
    PAPER_STRATEGIES, BIND_THRESH, diagnose_stale_directory_names,
    known_population, raw_table, warm_cache,
)

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 50)

# Strategy label -> directory name, appendix order (all eleven arms).
APPENDIX_STRATEGIES = {
    "Instant": "binding_raw", "Instant\n+Hint": "binding_direct_hinted",
    "Reason": "binding_reason", "Reason\n+FirstLetter": "binding_reason_firstletter",
    "Reason\n+Hint": "binding_reason_hinted", "NoReason": "binding_no_reason",
    "Self-Ask": "binding_selfask", "Self-Ask\n+Binding": "binding_selfask_modified_2",
    "MC": "binding_mc_raw", "MC\n+Reason": "binding_mc_reason", "MC\n+NoReason": "binding_mc_no_reason",
}
# Fig. 2 / main paper: the six fully-covered arms (deliberately excludes the
# hinted / no-reason arms, only run for a subset of models).
FIG2_STRATEGIES = {
    "Instant": "binding_raw", "Reason": "binding_reason", "Self-Ask": "binding_selfask",
    "Self-Ask\n+Binding": "binding_selfask_modified_2", "MC": "binding_mc_raw", "MC\n+Reason": "binding_mc_reason",
}

warm_cache()
diagnose_stale_directory_names()


## Build the pooled (athletes + custom) failure-rate cells

Each cell is `(n_fail, n_eval, n_known)`: `n_fail`/`n_eval` over the strategy's
own recorded trials, `n_known` from the shared known population -- the ratio
`n_eval/n_known` is what makes a cell's coverage "ragged" in the appendix
heatmap (shaded) instead of full (Fig. 2).


In [ ]:
def pooled_cell(strategies, model_dir, binding_dir):
    """(failures, n_eval, n_known) pooled over athletes+custom for one model
    x strategy, or None if neither dataset has this strategy for this model."""
    failures = n_eval = n_known = 0
    seen = False
    for tag in DATASETS:
        entity_table, _ = raw_table(tag, model_dir)
        population = known_population(tag, model_dir, 0.6)
        if entity_table.empty or not population or binding_dir not in entity_table.columns:
            continue
        column = entity_table.loc[entity_table.index.isin(population), binding_dir].dropna()
        if not len(column):
            continue
        failures += int((column < BIND_THRESH).sum())
        n_eval += len(column)
        n_known += len(population)
        seen = True
    return (failures, n_eval, n_known) if seen else None


def build_heatmap_data(strategies):
    """List of rows (one per model), each a list of cells (one per strategy)."""
    return [[pooled_cell(strategies, model_dir, binding_dir) for binding_dir in strategies.values()]
            for model_dir in MODEL_DIRS]


def incomplete_coverage_cells(strategies):
    """(row, col) pairs where n_eval < n_known for that model x strategy --
    shaded in the appendix heatmap to mark ragged coverage."""
    incomplete = set()
    for row, model_dir in enumerate(MODEL_DIRS):
        for col, binding_dir in enumerate(strategies.values()):
            cell = pooled_cell(strategies, model_dir, binding_dir)
            if cell is None or cell[1] < cell[2]:
                incomplete.add((row, col))
    return incomplete


## Render the heatmap (ported from the original plotting notebook, unchanged)

All typography/geometry knobs live in `STYLE` below the function so you can
retune a font size and re-run just the last cell.


In [ ]:
DEFAULT_STYLE = dict(
    font_scale=1.0, pct_fontsize=30, sub_fontsize=20, xtick_fontsize=30, ytick_fontsize=30,
    na_fontsize=21, star_fontsize=30, cbar_label_fontsize=30, cbar_tick_fontsize=21,
    pct_dy=-0.14, sub_dy=0.24, star_dx=0.36, star_dy=-0.32,
    xtick_rotation=45, xtick_ha="right", ytick_rotation=45, ytick_ha="right",
    cell_height_in=1.62, cell_aspect=0.60, label_margin_in=5.6, cbar_margin_in=3.0,
    top_margin_in=3.4, cbar_thickness_in=0.42, cbar_pad_in=0.45,
    cbar_label="Failure Rate (%)", gridline_lw=1.5, shade_alpha=0.35, preview_dpi=45,
)


def figure_geometry(n_cols, n_rows, st):
    cell_w = st["cell_height_in"] / st["cell_aspect"]
    width = st["label_margin_in"] + n_cols * cell_w + st["cbar_margin_in"]
    height = st["top_margin_in"] + n_rows * st["cell_height_in"]
    return (width, height)


def render_heatmap(data, columns, star_column_idx, outpath, style=None, shaded_cells=None):
    st = {**DEFAULT_STYLE, **(style or {})}
    fs = lambda key: st[key] * st["font_scale"]
    shaded_cells = set(shaded_cells or [])
    n_rows, n_cols = len(ROWS), len(columns)
    figsize = figure_geometry(n_cols, n_rows, st)

    pct = np.full((n_rows, n_cols), np.nan)
    for i in range(n_rows):
        for j in range(n_cols):
            cell = data[i][j]
            if cell and cell[1] > 0:
                pct[i, j] = cell[0] / cell[1] * 100

    norm, cmap = Normalize(vmin=0, vmax=100), plt.cm.viridis_r
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(pct, cmap=cmap, norm=norm, interpolation="nearest", aspect=st["cell_aspect"])
    ax.set_xticks(np.arange(n_cols))
    ax.set_xticklabels(columns, fontsize=fs("xtick_fontsize"), rotation=st["xtick_rotation"], ha=st["xtick_ha"])
    ax.set_yticks(np.arange(n_rows))
    ax.set_yticklabels(ROWS, fontsize=fs("ytick_fontsize"), rotation=st["ytick_rotation"], ha=st["ytick_ha"])

    axes_width_in = n_cols * (st["cell_height_in"] / st["cell_aspect"])
    cbar = plt.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), ax=ax,
                        fraction=st["cbar_thickness_in"] / axes_width_in, pad=st["cbar_pad_in"] / axes_width_in)
    cbar.set_label(st["cbar_label"], fontsize=fs("cbar_label_fontsize"))
    cbar.ax.tick_params(labelsize=fs("cbar_tick_fontsize"))

    for row, col in shaded_cells:
        ax.add_patch(plt.Rectangle((col - 0.5, row - 0.5), 1, 1, facecolor="white",
                                   edgecolor="none", alpha=1.0 - st["shade_alpha"], zorder=1))

    for i in range(n_rows):
        for j in range(n_cols):
            cell, is_shaded = data[i][j], (i, j) in shaded_cells
            if cell is None:
                ax.text(j, i, "N/A", ha="center", va="center", color="0.55" if is_shaded else "gray",
                        fontsize=fs("na_fontsize"), style="italic", zorder=3)
                continue
            failures, n_eval, n_known = cell
            percentage = failures / n_eval * 100 if n_eval else 0
            r, g, b, _ = cmap(norm(percentage))
            text_color = "0.45" if is_shaded else ("white" if 0.2126*r+0.7152*g+0.0722*b < 0.4 else "black")
            ax.text(j, i + st["pct_dy"], f"{percentage:.0f}%", ha="center", va="center",
                    color=text_color, fontsize=fs("pct_fontsize"), zorder=3)
            ax.text(j, i + st["sub_dy"], f"({n_eval}/{n_known})", ha="center", va="center",
                    color=text_color, fontsize=fs("sub_fontsize"), zorder=3)

    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.grid(False)
    for i in range(n_rows):
        for j in range(n_cols):
            ax.add_patch(plt.Rectangle((j - 0.5, i - 0.5), 1, 1, fill=False, lw=st["gridline_lw"], edgecolor="white"))

    if star_column_idx is not None:
        for row, model_label in enumerate(ROWS):
            if model_label in STAR_MODELS:
                ax.text(star_column_idx + st["star_dx"], row + st["star_dy"], "★",
                        fontsize=fs("star_fontsize"), ha="center", va="center",
                        color="0.45" if (row, star_column_idx) in shaded_cells else "black", zorder=3)

    plt.tight_layout()
    if outpath:
        plt.savefig(outpath, format="pdf", bbox_inches="tight", facecolor="white", pad_inches=0.3)
        print(f"  saved -> {outpath}  ({n_cols}x{n_rows} cells)")
    fig.set_dpi(st["preview_dpi"])
    return fig


## Generate both figures


In [ ]:
WHICH = "both"     # "fig2" | "appendix" | "both"
SAVE_PDF = True

STYLE = dict(DEFAULT_STYLE)          # copy and override individual keys here to retune
STYLE_APPENDIX = {**STYLE, "font_scale": STYLE["font_scale"] * 1.0}

if WHICH in ("fig2", "both"):
    columns = list(FIG2_STRATEGIES)
    reason_idx = columns.index("Reason")
    render_heatmap(build_heatmap_data(FIG2_STRATEGIES), columns, reason_idx,
                  outpath="fig_2_binding_failures.pdf" if SAVE_PDF else None, style=STYLE)
    plt.show()

if WHICH in ("appendix", "both"):
    columns = list(APPENDIX_STRATEGIES)
    reason_idx = columns.index("Reason")
    shaded = incomplete_coverage_cells(APPENDIX_STRATEGIES)
    if shaded:
        print(f"  shading {len(shaded)} incomplete-coverage cells")
    render_heatmap(build_heatmap_data(APPENDIX_STRATEGIES), columns, reason_idx,
                  outpath="appendix_b4_binding_failures.pdf" if SAVE_PDF else None,
                  style=STYLE_APPENDIX, shaded_cells=shaded)
    plt.show()
